# FR Fig. 08 prev - Annual Operation

This notebook runs the annual FR service simulation for the selected BESS solution. For annual simulations, `SoC_i = 0.5`. The main output is `Annual_Op_<BatteryNode>_<E_BESS>kWh_<P_BESS>kW.csv`.


In [ ]:
from pathlib import Path
from datetime import datetime
import os
import sys
import time
import pandas as pd

sys.path.append(os.getcwd())
from f_BESS_Sv_FR import f_BESS_Sv_FR

script_dir = Path.cwd()
print(f'script_dir: {script_dir}')
print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)


In [ ]:
Svc                  = 'FR'

n_dc                 = 0.9
n_ch                 = 0.9
primera_hora         = 1
ultima_hora          = 8760
annual_discount_rate = 0.02

BatteryNode          = 'bus2017'
E_BESS               = 2000
P_BESS               = 500
SoC_i                = 0.5  # Annual simulations start from 50% SoC

annual_output = script_dir / f'Annual_Op_{BatteryNode}_{E_BESS}kWh_{P_BESS}kW.csv'

print('##' * 40)
print(f'Service: {Svc}')
print(f'BatteryNode: {BatteryNode}')
print(f'P_BESS: {P_BESS} kW')
print(f'E_BESS: {E_BESS} kWh')
print(f'Initial SoC: {SoC_i}')
print(f'Hours: {primera_hora} to {ultima_hora}')
print(f'Annual output: {annual_output}')
print('##' * 40)


In [ ]:
start_time = time.time()
df_info_STK, annual_revenue, f1, f2, f3, f4, f5, f6, f_i_1, f_i_2, f_i_3 = f_BESS_Sv_FR(
    P_BESS,
    E_BESS,
    n_dc,
    n_ch,
    BatteryNode,
    primera_hora,
    ultima_hora,
    annual_discount_rate,
    SoC_i,
)
elapsed_time = (time.time() - start_time) / 60
print(f'Elapsed Time: {elapsed_time:.2f} minutes')


In [ ]:
expected_columns = [
    'hr', 'Col_SoC', 'P_BESS_t', 'name_policy', 'P_3f_cb_302',
    'P_3f_cb_302_BESS_0', 'DSS_Total_Looses', 'switch_FR', 'P_bid_h',
    'switch_PS', 'PS_peak', 'SW_FR_SoCmin', 'SW_FR_SoCmax',
    'DSS_Total_Looses_BESS', 'f4_t', 'ISV', 'ICV', 'm_p_I2_I1',
    'm_p_I2_I1_BESS', 'm_p_I0_I1', 'm_p_I0_I1_BESS',
    'm_p_Normal', 'm_p_Normal_BESS', 'f3_t', 'f_i_1_t',
    'f_i_2_t', 'f_i_3_t',
]
missing_columns = [col for col in expected_columns if col not in df_info_STK.columns]
if missing_columns:
    raise ValueError(f'Missing expected annual-operation columns: {missing_columns}')

df_info_STK = df_info_STK[expected_columns]
df_info_STK.to_csv(annual_output, index=False)
print(f'Annual CSV saved to: {annual_output}')
print(df_info_STK.head())
